# Tunix + Gemma Reasoning (SFT + GRPO)

This notebook shows how to fine-tune Gemma to 'show its work' using Tunix on TPU.

References: Tunix (JAX, TPU-native) and Gemma models on Hugging Face.

In [ ]:
# Environment setup (run on GCP TPU VM)
!pip -q install -U pip wheel
!pip -q install tunix transformers datasets sentencepiece accelerate wandb rich


In [ ]:
import os, json, yaml, pathlib
print('Working dir:', os.getcwd())
# Select TPU size: default v4-8; set TPU_SIZE='v4-16' to switch
TPU_SIZE = os.environ.get('TPU_SIZE', 'v4-8')
CFG_SFT = f"configs/sft_gemma2_2b_{'v4_8' if TPU_SIZE=='v4-8' else 'v4_16'}.yaml"
CFG_GRPO = f"configs/grpo_gemma2_2b_{'v4_8' if TPU_SIZE=='v4-8' else 'v4_16'}.yaml"
print('TPU_SIZE:', TPU_SIZE)
print('Using SFT config:', CFG_SFT)
print('Using GRPO config:', CFG_GRPO)
cfg_sft = yaml.safe_load(open(CFG_SFT))
cfg_grpo = yaml.safe_load(open(CFG_GRPO))


## Data preparation (GSM8K)

In [ ]:
# Prepare GSM8K data
!python -m src.tunix_reasoning.data --dataset gsm8k --split train --out data/processed/gsm8k_train.jsonl
!wc -l data/processed/gsm8k_train.jsonl


## Supervised Fine-Tuning (SFT) with Tunix

In [ ]:
# Prepare a derived config referencing our processed data
!python -m src.tunix_reasoning.train_sft --config {CFG_SFT} --data data/processed/gsm8k_train.jsonl


Run Tunix SFT trainer here (refer to Tunix examples).
Make sure to point the trainer to the derived config and data file paths.

## GRPO Reinforcement Learning with Reward Composition

In [ ]:
# Prepare a derived config for GRPO, initializing from SFT checkpoint
!python -m src.tunix_reasoning.train_grpo --config {CFG_GRPO} --data data/processed/gsm8k_train.jsonl --init_checkpoint checkpoints/sft_last


Run Tunix GRPO trainer here (refer to Tunix GRPO demo notebook).
This uses the composite reward from src/tunix_reasoning/rewards.py.

## Eval and Inference

In [ ]:
# Quick eval (uses Transformers to load a saved checkpoint)
# Replace checkpoints/grpo_best with your final checkpoint path
!python -m src.tunix_reasoning.eval --config configs/eval.yaml --checkpoint checkpoints/grpo_best || echo 'Set checkpoint first'


In [ ]:
# Inference on a custom question
!python -m src.tunix_reasoning.inference --checkpoint checkpoints/grpo_best --question 'If Alice has 3 apples and buys 5 more, how many apples does she have?' || echo 'Set checkpoint first'
